# Day 6: controlled NVIDIA serving investigation
Run only in a selected Colab GPU runtime. This notebook uses a committed source archive and loopback vLLM server. Save every JSON and profiler artifact; no result is prefilled.


In [ ]:
import hashlib, json, pathlib, platform, subprocess, sys, zipfile
from google.colab import files
EXPECTED_SHA256 = ""  # paste local shasum of git archive
assert len(EXPECTED_SHA256) == 64
name, data = next(iter(files.upload().items()))
assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256
source_dir = pathlib.Path('/content/jev-day6-source')
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as archive:
    assert all(not pathlib.PurePosixPath(p).is_absolute() and '..' not in pathlib.PurePosixPath(p).parts for p in archive.namelist())
    archive.extractall(source_dir)
print(platform.platform(), sys.version)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## Install and resolve actual model identity
The pinned vLLM version is a starting candidate. Consult the current vLLM GPU installation guide against this runtime before installing. Record any incompatibility rather than silently changing versions.


In [ ]:
VLLM_VERSION = '0.18.0'
subprocess.run([sys.executable, '-m', 'pip', 'install', str(source_dir), f'vllm=={VLLM_VERSION}'], check=True)
from huggingface_hub import HfApi
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
MODEL_REVISION = HfApi().model_info(MODEL).sha
assert MODEL_REVISION
config = json.loads((source_dir / 'config/day6-serving-example.json').read_text())
config['model_revision'] = MODEL_REVISION
config['tokenizer_revision'] = MODEL_REVISION
config_path = pathlib.Path('/content/day6-serving.json')
config_path.write_text(json.dumps(config, indent=2))
print(config)


## Start loopback server and record environment
Use the same model revision and settings in every baseline/candidate run. Check server readiness and actual allocation before benchmarking.


In [ ]:
import os, time, urllib.request
log = open('/content/day6-vllm.log', 'w')
server = subprocess.Popen(['vllm', 'serve', MODEL, '--revision', MODEL_REVISION, '--tokenizer-revision', MODEL_REVISION, '--host', '127.0.0.1', '--port', '8000', '--dtype', config['dtype'], '--max-model-len', str(config['max_model_len']), '--max-num-seqs', str(config['max_num_seqs']), '--max-num-batched-tokens', str(config['max_num_batched_tokens']), '--no-enable-prefix-caching'], stdout=log, stderr=subprocess.STDOUT)
for attempt in range(120):
    if server.poll() is not None: raise RuntimeError('vLLM exited; inspect /content/day6-vllm.log')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).read()
        break
    except Exception: time.sleep(2)
else: raise TimeoutError('vLLM readiness timeout')
subprocess.run([sys.executable, '-m', 'jev_gateway.day6.environment', '--source-archive', name, '--config', str(config_path), '--model', MODEL, '--model-revision', MODEL_REVISION, '--dtype', config['dtype'], '--output', '/content/day6-environment.json'], check=True)


## Synthetic performance baseline
These prompts contain no hidden quality-test items. This is a warm, loopback, burst-arrival, four-concurrency client run. Repeat with new output paths and record run order. Profiling is a separate diagnostic run.


In [ ]:
subprocess.run([sys.executable, '-m', 'jev_gateway.day6.workload', '--output', '/content/day6-workload.json'], check=True)
for index in range(3):
    subprocess.run([sys.executable, '-m', 'jev_gateway.day6.benchmark', '--workload', '/content/day6-workload.json', '--config', str(config_path), '--output', f'/content/day6-baseline-{index}.json', '--label', 'baseline', '--concurrency', '4', '--warmup', '4'], check=True)


## Optional diagnostic profile and cleanup
Run Nsight only if installed and permitted. A profile must not be mixed into headline timing runs. Copy all artifacts and the server log before the runtime disconnects.


In [ ]:
import shutil
if shutil.which('nsys'):
    subprocess.run([sys.executable, '-m', 'jev_gateway.day6.profile', '--output-dir', '/content/day6-profile', '--', sys.executable, '-m', 'jev_gateway.day6.benchmark', '--workload', '/content/day6-workload.json', '--config', str(config_path), '--output', '/content/day6-profile-diagnostic.json', '--label', 'profile-diagnostic'], check=True)
else:
    print('nsys unavailable: no GPU trace collected')
server.terminate()
server.wait(timeout=20)
log.close()
